In [1]:
import huggingface_hub as hfh
from datasets import load_dataset, get_dataset_split_names
hfh.login()

dsname = 'cornell-movie-review-data/rotten_tomatoes'

full_dataset = load_dataset(path=dsname, cache_dir='./mydatasets')
splits = full_dataset.keys()

/home/jonathan/venvs/dl/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
trainset = full_dataset['train']
validset = full_dataset['validation']

In [3]:
import numpy as np

for i in np.random.permutation(len(trainset))[:5]:
    row = trainset[i]
    print(f'{row['label']}: {row['text']}')

1: . . . a haunting vision , with images that seem more like disturbing hallucinations .
0: the good thing -- the only good thing -- about extreme ops is that it's so inane that it gave me plenty of time to ponder my thanksgiving to-do list .
0: opens as promising as any war/adventure film you'll ever see and dissolves into a routine courtroom drama , better suited for a movie titled " glory : a soldier's story . "
1: compared to his series of spectacular belly flops both on and off the screen , runteldat is something of a triumph .
1: confounding because it solemnly advances a daringly preposterous thesis . acting cannot be acted .


In [4]:
from transformers import AutoTokenizer, AutoModel
def model_tokenizer():
    slug = 'distilbert/distilbert-base-uncased'
    return AutoModel.from_pretrained(slug), AutoTokenizer.from_pretrained(slug)

model, tokenizer = model_tokenizer()
model.eval()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11276.53it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [5]:
if False:
    sentence = ['this is a sample sentence', 'aight imma head out']
    tokens = tokenizer.encode(list(trainset['text']), return_tensors='pt', padding=True)
    validtokens = tokenizer.encode(list(validset['text']), return_tensors='pt', padding=True)
    output = model(tokens)

In [6]:
from transformers import pipeline

extractor = pipeline('feature-extraction', model=model, tokenizer=tokenizer, batch_size=32, device='cuda')
train_feats = extractor(list(trainset['text']), return_tensors='pt')
valid_feats = extractor(list(validset['text']), return_tensors='pt')

In [7]:
import numpy as np

train_feats = np.vstack([f[0][0] for f in train_feats])
valid_feats = np.vstack([f[0][0] for f in valid_feats])
train_lab = np.array(trainset['label'])
valid_lab = np.array(validset['label'])

In [8]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

sm = MLPClassifier()
sm = sm.fit(train_feats, train_lab)
print('TRAIN')
preds = sm.predict(train_feats)
print(classification_report(train_lab, preds))

print('VALID')
preds = sm.predict(valid_feats)
print(classification_report(valid_lab, preds))

TRAIN
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4265
           1       1.00      1.00      1.00      4265

    accuracy                           1.00      8530
   macro avg       1.00      1.00      1.00      8530
weighted avg       1.00      1.00      1.00      8530

VALID
              precision    recall  f1-score   support

           0       0.79      0.75      0.77       533
           1       0.77      0.80      0.78       533

    accuracy                           0.78      1066
   macro avg       0.78      0.78      0.78      1066
weighted avg       0.78      0.78      0.78      1066



/home/jonathan/venvs/dl/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
